> **Disclaimer.** This project is educational and is not investment advice. Backtest results are historical and do not predict future results.

# Turtle on the stocks that looked interesting: a closer look

`turtle_stocks.ipynb` ranked 30 liquid Ibovespa names under the Turtle strategy's fixed setting (`docs/brief.md` section 7c, realistic `next_open` fill, 0.03% on every fill). Seven of them beat the CDI over the full history and were positive in at least two of three periods. This notebook looks at those seven, plus **PETR4 and VALE3** as references, in more detail: portfolio view, charts, per-stock statistics and trades. A dropdown at the end lets you open **any** of the candidate tickers.

**Read this before the charts:**
- **These stocks were chosen *because* they did well on this same history.** Every number below is therefore optimistic by construction. Section 8 shows how much: it compares the shortlist with the rest, and runs the same kind of pick using only the early data. `turtle_stocks.ipynb` found that past profit did not reliably predict later profit over long horizons.
- **No capital cap.** `capital_brl` only sets the risk budget, so the money in a position is a multiple of it (`peak money / capital`). Returns are on `capital_brl`, and some names lost more than the whole capital.
- **Survivorship and one regime.** The tickers are today's liquid names from a draft list written from memory (not B3's verified index composition). About 14 years of one market.
- **Charts need a Jupyter front end.** The Plotly charts are interactive in JupyterLab or in an HTML export (`jupyter nbconvert --to html`). GitHub does not render them. Prices are shown **rebased to 100 and weekly**, never as candles, so the notebook publishes results and not the data.
- **Not modelled:** taxes, slippage beyond the 0.03% fee, dividends (raw closes), a shared-cash rule for simultaneous signals.

In [1]:
import json
import logging
import urllib.request
import warnings
from dataclasses import replace
from datetime import UTC, date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
from IPython.display import display
from plotly.subplots import make_subplots

from b3core.benchmark import compare_with_index, index_equity
from b3core.indicators import atr
from b3core.portfolio import buy_and_hold_equity, combine_equity, summarize_portfolio
from b3core.turtle import TurtleConfig, run_turtle
from b3core.universe import scan_universe

pd.options.display.float_format = "{:,.4f}".format
CACHE = Path("../samples/ifr2_universe")  # gitignored vendor data, shared with the IFR2 notebooks
CACHE.mkdir(parents=True, exist_ok=True)

logging.getLogger("yfinance").setLevel(logging.CRITICAL)  # failed downloads are listed below, once

In [2]:
print(f"Run on {date.today():%Y-%m-%d}. Prices are raw Yahoo Finance closes up to the latest session; the CDI source is printed in section 3.")

Run on 2026-10-21. Prices are raw Yahoo Finance closes up to the latest session; the CDI source is printed in section 3.


## 1. Parameters

`SHORTLIST` is the seven names from `turtle_stocks.ipynb` section 10 (the code checks them against the rule in section 8). `REFERENCE` is PETR4 and VALE3, added by the owner. `IBOV_CANDIDATES` is the draft list of candidates for the dropdown.

In [3]:
CONFIG = replace(TurtleConfig.preset(), fill_at="next_open")  # script setting, realistic fill, 0.03% on every fill
START = date(2012, 1, 1)
END = date(2027, 1, 1)  # exclusive

# Draft candidates, from memory: edit freely. Tickers that fail to download are reported below.
IBOV_CANDIDATES = [
    "PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "ABEV3", "WEGE3", "RENT3", "LREN3", "MGLU3",
    "SUZB3", "GGBR4", "CSNA3", "USIM5", "JBSS3", "RADL3", "RAIL3", "EQTL3", "CMIG4", "ELET3",
    "TOTS3", "PRIO3", "CSAN3", "BBSE3", "SBSP3", "VIVT3", "B3SA3", "BRAP4", "GOAU4", "CYRE3",
    "ITSA4", "KLBN11", "TAEE11", "SANB11", "BPAC11", "HAPV3", "COGN3", "BEEF3", "MRFG3", "CPLE6",
    "EGIE3", "SMTO3", "RDOR3", "HYPE3", "CCRO3", "UGPA3", "VBBR3", "EMBR3", "ALPA4", "MULT3",
    "BRFS3", "ASAI3", "GMAT3", "CMIN3", "ENEV3", "RRRP3",
]
N_LIQUID_IBOV = 100  # no liquidity cut: every candidate with data is a choice in the dropdown
N_LIQUID_BDR = 10  # the BDR universe: the ten most traded of the candidates
MIN_CANDLES_IBOV = 750  # about three years of daily candles
LIQUIDITY_WINDOW = 250  # candles used to measure the median traded value
SHORTLIST = ["BPAC11", "PRIO3", "MGLU3", "WEGE3", "ENEV3", "RADL3", "RENT3"]  # turtle_stocks.ipynb, section 10
REFERENCE = ["PETR4", "VALE3"]  # added by the owner, not selected by any rule
NINE = SHORTLIST + REFERENCE

## 2. Load candles

Daily candles from Yahoo Finance, raw prices, one CSV per ticker in the shared cache folder.

In [4]:
def load_candles(ticker: str) -> pd.DataFrame | None:
    path = CACHE / f"{ticker}.csv"
    if not path.exists():
        raw = yf.download(f"{ticker}.SA", start=START, end=END, auto_adjust=False, progress=False)
        if raw.empty:
            return None
        raw.columns = raw.columns.get_level_values(0)
        raw = raw.dropna(subset=["Open", "High", "Low", "Close"])
        raw[["Open", "High", "Low", "Close", "Volume"]].to_csv(path)
    raw = pd.read_csv(path, index_col=0, parse_dates=True)
    return pd.DataFrame(
        {
            "ts": raw.index.tz_localize("America/Sao_Paulo").tz_convert("UTC"),
            "open": raw["Open"].to_numpy(),
            "high": raw["High"].to_numpy(),
            "low": raw["Low"].to_numpy(),
            "close": raw["Close"].to_numpy(),
            "volume": raw["Volume"].to_numpy(dtype=float),
        }
    ).reset_index(drop=True)


def load_ibov() -> pd.Series:
    path = CACHE / "ibov.csv"
    if not path.exists():
        raw = yf.download("^BVSP", start=START, end=END, auto_adjust=False, progress=False)
        raw.columns = raw.columns.get_level_values(0)
        raw[["Close"]].dropna().to_csv(path)
    return pd.read_csv(path, index_col=0, parse_dates=True)["Close"].dropna()


def local_day(candles: pd.DataFrame) -> pd.DatetimeIndex:
    return pd.DatetimeIndex(candles["ts"]).tz_convert("America/Sao_Paulo").tz_localize(None).normalize()


ibov = load_ibov()


def load_universe(tickers: list[str], min_candles: int) -> tuple[dict, dict]:
    """Raw candles per ticker, and the tickers left out (no data, or too little history)."""
    loaded = {t: load_candles(t) for t in tickers}
    out = {
        "no data": [t for t, c in loaded.items() if c is None],
        "too short": [t for t, c in loaded.items() if c is not None and len(c) < min_candles],
    }
    data = {t: c for t, c in loaded.items() if c is not None and len(c) >= min_candles}
    return data, out


ibov_all, ibov_out = load_universe(IBOV_CANDIDATES, MIN_CANDLES_IBOV)
print(f"Ibovespa candidates: {len(ibov_all)} loaded; left out: {ibov_out}")
print(f"IBOV: {len(ibov)} closes, {ibov.index[0].date()} to {ibov.index[-1].date()}")

Ibovespa candidates: 48 loaded; left out: {'no data': ['JBSS3', 'ELET3', 'MRFG3', 'CPLE6', 'CCRO3', 'EMBR3', 'BRFS3', 'RRRP3'], 'too short': []}
IBOV: 3667 closes, 2012-01-03 to 2026-10-16


## 3. The CDI

The benchmark is the daily CDI compounded over each window (`docs/cdi.md`). The Central Bank API is tried first; if it is blocked, the **Selic target minus 0.10 points a year** is used, which is an approximation (the cell prints which one). A file `samples/ifr2_universe/cdi.csv` (series 12, columns `date` and `rate`) overrides both.

In [5]:
def fetch(url: str) -> str:
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read().decode("utf-8")


trading_days = sorted(set().union(*[set(local_day(c)) for c in ibov_all.values()]))


def cdi_from_bcb() -> pd.Series:
    rows, begin = [], START
    while begin < END and begin < datetime.now(UTC).date():
        stop = min(begin + timedelta(days=3_000), END - timedelta(days=1))  # about 8 years
        url = (
            "https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados"
            f"?formato=json&dataInicial={begin:%d/%m/%Y}&dataFinal={stop:%d/%m/%Y}"
        )
        rows += json.loads(fetch(url))  # raises when the API answers with an HTML error page
        begin = stop + timedelta(days=1)
    index = pd.to_datetime([r["data"] for r in rows], format="%d/%m/%Y")
    return pd.Series([float(r["valor"]) for r in rows], index=index).sort_index()


def cdi_from_selic_target() -> pd.Series:
    url = "http://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='BM366_TJOVER366')"
    values = json.loads(fetch(url))["value"]
    annual = pd.Series(
        [v["VALVALOR"] for v in values], index=pd.to_datetime([v["VALDATA"][:10] for v in values])
    )
    daily = (1 + (annual - 0.10) / 100) ** (1 / 252) - 1  # percent a year to a business-day rate
    return (daily * 100).reindex(trading_days).dropna()  # trading days only


def load_cdi() -> tuple[pd.Series, str]:
    exact = CACHE / "cdi.csv"
    approx = CACHE / "cdi_from_selic_target.csv"
    if not exact.exists() and not approx.exists():
        try:
            cdi_from_bcb().rename_axis("date").rename("rate").to_csv(exact, date_format="%d/%m/%Y")
        except Exception as error:  # the API blocks some networks
            print(f"Central Bank API unavailable ({type(error).__name__}); using the Selic target")
            cdi_from_selic_target().rename_axis("date").rename("rate").to_csv(approx, date_format="%d/%m/%Y")
    path = exact if exact.exists() else approx
    frame = pd.read_csv(path)
    index = pd.to_datetime(frame["date"], format="%d/%m/%Y")
    series = pd.Series(frame["rate"].to_numpy(dtype=float), index=index).sort_index()
    label = "CDI (BCB series 12)" if path == exact else "Selic target minus 0.10 points (approximation)"
    return series, label


cdi, CDI_SOURCE = load_cdi()
print(f"source: {CDI_SOURCE}")
print(f"{len(cdi)} days, {cdi.index[0].date()} to {cdi.index[-1].date()}")

source: CDI (BCB series 12)
3715 days, 2012-01-02 to 2026-10-16


## 4. Clean the candles and build the universe

Candles before the first IBOV close, and candles on days with no CDI rate (Yahoo's zero-volume holiday rows and the days after the latest published rate), are **dropped and printed**, never filled. The universe is then the `N_LIQUID_IBOV` most traded names: the median of `close * volume` over the last `LIQUIDITY_WINDOW` candles, a survivor ranking on recent data.

In [6]:
def clean(data: dict) -> tuple[dict, pd.DataFrame]:
    cleaned, rows = {}, {}
    cdi_days = set(pd.DatetimeIndex(cdi.index).normalize())
    for t, c in data.items():
        days = local_day(c)
        before_ibov = days < ibov.index[0]
        no_rate = ~before_ibov & ~days.isin(cdi_days) & (days > ibov.index[0])
        zero_volume = no_rate & (c["volume"].to_numpy() == 0)
        keep = ~before_ibov & ~no_rate
        cleaned[t] = c[keep].reset_index(drop=True)
        rows[t] = {
            "before first IBOV close": int(before_ibov.sum()),
            "no CDI rate": int(no_rate.sum()),
            "...of which zero volume": int(zero_volume.sum()),
        }
    return cleaned, pd.DataFrame(rows).T


def traded_value(data: dict) -> pd.Series:
    return pd.Series(
        {t: float((c["close"] * c["volume"]).tail(LIQUIDITY_WINDOW).median()) for t, c in data.items()}
    ).sort_values(ascending=False)


ibov_clean, ibov_dropped = clean(ibov_all)
dropped = ibov_dropped
print(f"{int((dropped > 0).any(axis=1).sum())} of {len(dropped)} tickers had rows dropped. Most dropped for one ticker:")
print(dropped.max().to_string())

ibov_liquidity = traded_value(ibov_clean)
ibov_data = {t: ibov_clean[t] for t in ibov_liquidity.index[:N_LIQUID_IBOV]}
print(f"Ibovespa universe ({len(ibov_data)}): {sorted(ibov_data)}")
ibov_liquidity.rename("median traded value (R$)").to_frame()

48 of 48 tickers had rows dropped. Most dropped for one ticker:
before first IBOV close    1
no CDI rate                9
...of which zero volume    6
Ibovespa universe (48): ['ABEV3', 'ALPA4', 'ASAI3', 'B3SA3', 'BBAS3', 'BBDC4', 'BBSE3', 'BEEF3', 'BPAC11', 'BRAP4', 'CMIG4', 'CMIN3', 'COGN3', 'CSAN3', 'CSNA3', 'CYRE3', 'EGIE3', 'ENEV3', 'EQTL3', 'GGBR4', 'GMAT3', 'GOAU4', 'HAPV3', 'HYPE3', 'ITSA4', 'ITUB4', 'KLBN11', 'LREN3', 'MGLU3', 'MULT3', 'PETR4', 'PRIO3', 'RADL3', 'RAIL3', 'RDOR3', 'RENT3', 'SANB11', 'SBSP3', 'SMTO3', 'SUZB3', 'TAEE11', 'TOTS3', 'UGPA3', 'USIM5', 'VALE3', 'VBBR3', 'VIVT3', 'WEGE3']


,median traded value (R$)
PETR4,"1,655,303,559.2301"
VALE3,"1,500,479,277.6146"
ITUB4,"1,006,870,459.4933"
BBDC4,"602,947,044.5090"
B3SA3,"583,429,987.9042"
BBAS3,"515,754,926.5521"
BPAC11,"503,176,801.0899"
PRIO3,"483,600,531.4396"
SBSP3,"415,292,526.3307"
ABEV3,"384,428,898.6681"


## 5. Run the strategy

The strategy runs once per ticker over its whole history (`runs`). The scan adds the benchmarks (`full`): the CDI, the IBOV and buying and holding the ticker. The engine's warnings (skipped units, several adds due on one candle) would print once per run, so they are silenced and counted in the statistics instead.

In [7]:
def run_all(data: dict, config: TurtleConfig) -> dict:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return {t: run_turtle(c, config) for t, c in data.items()}


def scan_one(data: dict, config: TurtleConfig, min_candles: int) -> pd.DataFrame:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return scan_universe(data, config, cdi, min_candles=min_candles, index_closes=ibov, runner=run_turtle)


universe = ibov_data
days_ = {t: local_day(c) for t, c in universe.items()}
all_days = sorted(set().union(*days_.values()))
MIN_CANDLES = 300  # about a year and a half of daily candles
CAPITAL = CONFIG.capital_brl

runs = run_all(universe, CONFIG)
full = scan_one(universe, CONFIG, MIN_CANDLES)
print(f"{len(universe)} tickers, {all_days[0].date()} to {all_days[-1].date()}; scanned: {len(full)}; skipped: {full.attrs['skipped'] or 'none'}")

48 tickers, 2012-01-03 to 2026-10-16; scanned: 48; skipped: none


## 6. The nine at a glance

`SHORTLIST` marks the seven selected on this history, `REFERENCE` the two added. Columns: results on `capital_brl`, the CDI and IBOV comparisons (`excess` is the strategy CAGR minus the CDI's over the same window), the equity drawdown, and how much money the strategy needed (`peak money / capital`).

In [8]:
def trades_frame(t: str) -> pd.DataFrame:
    rows = [
        {
            "entry": days_[t][x.entry_index], "exit": days_[t][x.exit_index], "units": x.units, "shares": x.qty,
            "avg entry": x.entry, "exit price": x.exit, "pnl (R$)": x.pnl, "pnl / capital": x.pnl / CAPITAL,
            "days": x.days, "reason": x.exit_reason,
        }
        for x in runs[t].trades
    ]
    return pd.DataFrame(rows)


def equity_of(t: str) -> pd.Series:
    return pd.Series(runs[t].equity, index=days_[t])


def drawdown_of(equity: pd.Series, capital: float) -> pd.Series:
    """Fall below the running peak of the equity, as a fraction; the curve starts at the capital."""
    peak = np.maximum.accumulate(np.concatenate(([capital], equity.to_numpy())))[1:]
    return pd.Series((peak - equity.to_numpy()) / peak, index=equity.index)


def scoreboard(names: list[str]) -> pd.DataFrame:
    table = full.reindex(names)[["n_trades", "cagr", "excess", "beats_cdi", "bh_cagr", "beats_bh", "ibov_cagr", "beats_ibov", "max_equity_drawdown", "calmar", "score_v1"]].copy()
    table["peak money / capital"] = pd.Series({t: runs[t].peak_cost / CAPITAL for t in names if t in runs})
    table["avg units"] = pd.Series({t: trades_frame(t)["units"].mean() for t in names if t in runs})
    table["lost more than the capital"] = pd.Series({t: bool(runs[t].equity.min() <= 0) for t in names if t in runs})
    table.insert(0, "group", ["shortlist" if t in SHORTLIST else "reference" for t in names])
    return table


print("shortlist tickers with no data:", [t for t in NINE if t not in universe] or "none")
nine = [t for t in NINE if t in universe]
scoreboard(nine)

shortlist tickers with no data: none


,group,n_trades,cagr,excess,beats_cdi,bh_cagr,beats_bh,ibov_cagr,beats_ibov,max_equity_drawdown,calmar,score_v1,peak money / capital,avg units,lost more than the capital
BPAC11,shortlist,31,0.2109,0.1190,True,0.3406,False,0.1238,True,0.3944,0.5347,33.0328,3.5610,2.4194,False
PRIO3,shortlist,58,0.1716,0.0736,True,0.1400,True,0.0877,True,0.5853,0.2933,42.8409,3.3728,2.1552,False
MGLU3,shortlist,47,0.1615,0.0635,True,0.1040,True,0.0877,True,0.4660,0.3466,39.6753,2.6933,2.5745,False
WEGE3,shortlist,53,0.1386,0.0405,True,0.2112,False,0.0877,True,0.2998,0.4622,41.5208,4.7263,2.6604,False
ENEV3,shortlist,48,0.1149,0.0169,True,-0.1118,True,0.0877,True,1.1286,0.1018,40.0000,4.5880,1.8958,True
RADL3,shortlist,59,0.1036,0.0056,True,0.1524,False,0.0877,True,0.3833,0.2702,38.3464,4.2279,2.3051,False
RENT3,shortlist,57,0.0981,0.0001,True,0.1321,False,0.0877,True,0.4509,0.2176,42.4526,3.2029,2.2456,False
PETR4,reference,66,0.0321,-0.0659,False,0.0685,False,0.0877,False,0.9207,0.0349,23.0505,4.4546,2.4545,False
VALE3,reference,68,0.0773,-0.0207,False,0.0301,True,0.0877,False,1.2182,0.0635,30.9443,5.6946,2.1618,True


## 7. The nine as one portfolio

The capital is split equally between the nine (independent slots of `capital / 9`, `b3core.portfolio`; each slot keeps `risk_pct` of its own capital and there is still no cap on the money in a slot). Compared with leaving the same money in the CDI, in the IBOV, and in buying and holding the nine equally.

In [9]:
slot = replace(CONFIG, capital_brl=CAPITAL / len(nine))
slot_runs = run_all({t: universe[t] for t in nine}, slot)
slot_curves = {t: pd.Series(r.equity, index=days_[t]) for t, r in slot_runs.items()}
portfolio = combine_equity(slot_curves)
summary = summarize_portfolio(portfolio, CAPITAL, cdi)
index_curve = index_equity(ibov, pd.DatetimeIndex(portfolio.index), CAPITAL)
held_curve = buy_and_hold_equity({t: universe[t] for t in nine}, CAPITAL)
held_curve.index = pd.DatetimeIndex(held_curve.index).tz_convert("America/Sao_Paulo").tz_localize(None).normalize() if getattr(held_curve.index, "tz", None) else held_curve.index
cdi_window = cdi[(cdi.index > portfolio.index[0]) & (cdi.index <= portfolio.index[-1])]
cdi_curve = CAPITAL * (1 + cdi_window / 100).cumprod()
cdi_curve = pd.concat([pd.Series([CAPITAL], index=[portfolio.index[0]]), cdi_curve])

weekly = lambda s: s.resample("W").last().dropna()
figure = go.Figure()
for label, curve, color in (("Turtle, the nine", portfolio, "#1f77b4"), ("CDI", cdi_curve, "#7f7f7f"), ("IBOV", index_curve, "#d62728"), ("Buy and hold, the nine", held_curve, "#2ca02c")):
    weekly_curve = weekly(curve)
    figure.add_trace(go.Scatter(x=weekly_curve.index, y=weekly_curve.round(0), name=label, line={"color": color, "width": 2 if label.startswith("Turtle") else 1.3}))
figure.update_layout(title="Equity: R$ 25,000 split between the nine, against the benchmarks (log scale)", yaxis={"type": "log", "title": "R$"}, height=480, hovermode="x unified")
figure.show()

print(f"Portfolio CAGR {summary.cagr:.1%}  |  CDI {summary.cdi_cagr:.1%}  |  excess {summary.excess:+.1%}  |  beats CDI: {summary.beats_cdi}")
print(f"Max equity drawdown {summary.max_equity_drawdown:.1%}  |  Calmar {summary.calmar:.2f}  |  sum of slot peaks / capital {sum(r.peak_cost for r in slot_runs.values()) / CAPITAL:.1f}x")
ibov_cmp = compare_with_index(summary.cagr, summary.years, ibov, portfolio.index)
print(f"IBOV CAGR {ibov_cmp.cagr:.1%}  |  beats IBOV: {ibov_cmp.beats}")

Portfolio CAGR 12.8%  |  CDI 9.8%  |  excess +3.0%  |  beats CDI: True
Max equity drawdown 17.3%  |  Calmar 0.74  |  sum of slot peaks / capital 4.0x
IBOV CAGR 8.8%  |  beats IBOV: True


In [10]:
drawdowns = {"Turtle, the nine": drawdown_of(portfolio, CAPITAL), "IBOV": drawdown_of(index_curve, CAPITAL), "Buy and hold, the nine": drawdown_of(held_curve, CAPITAL)}
figure = go.Figure()
for label, series in drawdowns.items():
    weekly_dd = series.resample("W").max().dropna()
    figure.add_trace(go.Scatter(x=weekly_dd.index, y=(-100 * weekly_dd).round(1), name=label, fill="tozeroy" if label.startswith("Turtle") else None))
figure.update_layout(title="Drawdown below the running peak (%)", yaxis={"title": "%"}, height=380, hovermode="x unified")
figure.show()

yearly_pnl = portfolio.resample("YE").last().diff()
yearly_pnl.iloc[0] = portfolio.resample("YE").last().iloc[0] - CAPITAL
yearly_return = (yearly_pnl / CAPITAL * 100).round(1)
colors = ["#2ca02c" if v >= 0 else "#d62728" for v in yearly_return]
figure = go.Figure(go.Bar(x=yearly_return.index.year, y=yearly_return, marker_color=colors, text=yearly_return.astype(str) + "%", textposition="outside"))
figure.update_layout(title="Result per calendar year (% of capital; the last year is partial)", height=360, xaxis={"type": "category"})
figure.show()

month_end = portfolio.resample("ME").last()
monthly = (month_end.diff().fillna(month_end - CAPITAL) / CAPITAL * 100)
grid = monthly.groupby([monthly.index.year, monthly.index.month]).first().unstack()
grid.columns = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"][: grid.shape[1]]
limit = float(np.nanmax(np.abs(grid.to_numpy())))
figure = go.Figure(go.Heatmap(z=grid.round(1).to_numpy(), x=list(grid.columns), y=[str(y) for y in grid.index], zmid=0, zmin=-limit, zmax=limit, colorscale="RdYlGn", colorbar={"title": "%"}))
figure.update_layout(title="Monthly result (% of capital)", height=460, yaxis={"autorange": "reversed"})
figure.show()

In [11]:
contribution = pd.DataFrame({
    "net pnl (R$)": {t: sum(x.pnl for x in r.trades) for t, r in slot_runs.items()},
    "trades": {t: len(r.trades) for t, r in slot_runs.items()},
    "peak money / capital": {t: r.peak_cost / CAPITAL for t, r in slot_runs.items()},
}).sort_values("net pnl (R$)", ascending=False)
contribution["share of the total"] = contribution["net pnl (R$)"] / contribution["net pnl (R$)"].sum()
figure = go.Figure(go.Bar(x=contribution.index, y=contribution["net pnl (R$)"].round(0), marker_color=["#2ca02c" if v >= 0 else "#d62728" for v in contribution["net pnl (R$)"]]))
figure.update_layout(title="Closed-trade profit by ticker inside the portfolio (R$)", height=340)
figure.show()
contribution

,net pnl (R$),trades,peak money / capital,share of the total
PRIO3,"26,086.8153",58,0.3724,0.2266
MGLU3,"22,330.5746",47,0.2956,0.1940
WEGE3,"16,229.5081",53,0.5248,0.1410
BPAC11,"14,758.7198",31,0.3902,0.1282
ENEV3,"11,100.2018",48,0.5096,0.0964
RADL3,"9,137.1025",59,0.4688,0.0794
RENT3,"8,252.9300",57,0.3549,0.0717
VALE3,"5,560.8633",68,0.6291,0.0483
PETR4,"1,673.7174",66,0.4921,0.0145


## 8. How much of this is selection?

The seven were picked from the same history they are scored on. Two checks:

1. **Groups.** The shortlist, the two references and every other candidate, full history. A shortlist that beats the rest is the selection showing, not proof it will keep doing so.
2. **A clean pick.** Use only the data **before** a cut date to choose the seven best by in-sample `score_v1` among the names that beat the CDI there, then score them **after** the cut, and compare with random picks of the same size (`pct of random`: the share of random seven-ticker sets with a lower median CAGR after the cut; 50% is chance).

In [12]:
group_of = lambda t: "shortlist" if t in SHORTLIST else ("reference" if t in REFERENCE else "other candidates")
groups = full.assign(group=[group_of(t) for t in full.index]).groupby("group").agg(
    tickers=("cagr", "size"), median_cagr=("cagr", "median"), beat_cdi=("beats_cdi", "sum"),
    median_excess=("excess", "median"), median_drawdown=("max_equity_drawdown", "median"),
)
groups

,tickers,median_cagr,beat_cdi,median_excess,median_drawdown
group,,,,,
other candidates,39,0.0415,4,-0.0565,0.6578
reference,2,0.0547,0,-0.0433,1.0694
shortlist,7,0.1386,7,0.0405,0.4509


In [13]:
rng = np.random.default_rng(0)  # seeded
rows = []
for frac in (0.5, 0.6, 0.7):
    cut = all_days[int(len(all_days) * frac)]
    before = {t: c[days_[t] < cut].reset_index(drop=True) for t, c in universe.items()}
    after = {t: c[days_[t] >= cut].reset_index(drop=True) for t, c in universe.items()}
    scan_before = scan_one(before, CONFIG, MIN_CANDLES)
    scan_after = scan_one(after, CONFIG, 150)["cagr"]
    eligible = scan_before[scan_before["beats_cdi"]].sort_values("score_v1", ascending=False)
    picked = [t for t in eligible.index[:7] if t in scan_after.index]
    pool = [t for t in scan_after.index if t in scan_before.index]
    random_medians = np.array([scan_after.loc[list(rng.choice(pool, 7, replace=False))].median() for _ in range(1_000)])
    chosen = scan_after.loc[picked].median()
    rows.append({
        "cut": cut.date(), "picked": ", ".join(picked), "picked: median cagr after": chosen,
        "all candidates: median cagr after": scan_after.loc[pool].median(), "pct of random": float((random_medians < chosen).mean()),
    })
pd.DataFrame(rows).set_index("cut")

,picked,picked: median cagr after,all candidates: median cagr after,pct of random
cut,,,,
2019-06-05,"USIM5, TAEE11, LREN3, RADL3, EQTL3, BEEF3, RENT3",-0.0206,-0.0206,0.4670
2020-11-25,"USIM5, TAEE11, GGBR4, RADL3, ALPA4, LREN3, BEEF3",-0.0487,-0.0723,0.5530
2022-05-23,"USIM5, BEEF3, TAEE11, CSNA3, WEGE3, LREN3, RENT3",0.0186,-0.0288,0.7050


## 9. Each of the nine

One block per ticker: rebased weekly price with the trade entries and exits, the equity curve, the drawdown, the profit of each trade, then the statistics and the best and worst trades. Triangles up are entries (at the average fill price of the units), crosses are exits (green a gain, red a loss; marker text shows units, shares and reason).

In [14]:
def pct(x: float) -> str:
    return f"{x:.1%}"


def stats_for(t: str) -> pd.DataFrame:
    result, row, tf = runs[t], full.loc[t], trades_frame(t)
    m = result.metrics
    payoff = f"{m.avg_gain / m.avg_loss:.2f}" if m.avg_loss > 0 else "n/a"
    items = [
        ("Period", f"{days_[t][0].date()} to {days_[t][-1].date()} ({len(days_[t])} candles)"),
        ("Closed trades", f"{m.n_trades}"),
        ("Win rate", pct(m.win_rate)),
        ("Average gain / average loss (R$)", f"{m.avg_gain:,.0f} / {m.avg_loss:,.0f}"),
        ("Payoff (gain / loss)", payoff),
        ("Profit factor", f"{m.profit_factor:.2f}"),
        ("Expected value per trade (% of capital)", pct(m.ev)),
        ("Best / worst streak (trades)", f"{m.best_streak} / {m.worst_streak}"),
        ("Net result (R$ and % of capital)", f"{m.net_pnl:,.0f}  ({pct(m.result)})"),
        ("CAGR", pct(row["cagr"])),
        ("CDI CAGR / excess", f"{pct(row['cdi_cagr'])} / {row['excess']:+.1%}"),
        ("Buy and hold CAGR / IBOV CAGR", f"{pct(row['bh_cagr'])} / {pct(row['ibov_cagr'])}"),
        ("Max equity drawdown / Calmar", f"{pct(row['max_equity_drawdown'])} / {row['calmar']:.2f}"),
        ("score_v1", f"{row['score_v1']:.1f}"),
        ("Units per trade (average / max)", f"{tf['units'].mean():.1f} / {tf['units'].max()}"),
        ("Days in a trade (average)", f"{tf['days'].mean():.0f}"),
        ("Exits by sell / by stop", f"{(tf['reason'] == 'sell').sum()} / {(tf['reason'] == 'stop').sum()}"),
        ("Peak money / capital", f"{result.peak_cost / CAPITAL:.1f}x"),
        ("Lost more than the capital", "yes" if result.equity.min() <= 0 else "no"),
        ("Open units at the end", f"{result.open_units}"),
        ("Skipped units / multi-add candles", f"{result.skipped_units} / {result.multi_add_candles}"),
    ]
    return pd.DataFrame(items, columns=["statistic", t]).set_index("statistic")


WEEK_MS = 7 * 24 * 3600 * 1000  # one week in milliseconds: the step of a weekly Plotly series


def weekly_series(series: pd.Series, how: str = "last") -> dict:
    """A weekly, evenly spaced series as Plotly's x0, dx and y: no date array to store."""
    resampled = getattr(series.resample("W"), how)().ffill()
    return {"x0": resampled.index[0].strftime("%Y-%m-%d"), "dx": WEEK_MS, "y": resampled}


def ticker_traces(t: str) -> list[tuple[go.Scatter | go.Bar, int]]:
    """The traces of one ticker and the chart row each one belongs to (1 price, 2 equity, 3 drawdown, 4 trades)."""
    close = pd.Series(universe[t]["close"].to_numpy(), index=days_[t])
    base = float(close.iloc[0])
    price = weekly_series(100 * close / base)
    equity = weekly_series(equity_of(t))
    drawdown = weekly_series(drawdown_of(equity_of(t), CAPITAL), "max")
    tf = trades_frame(t)
    wins = tf["pnl (R$)"] >= 0
    label = lambda r: f"{r['units']}u, {r['shares']} sh, {r['reason']}, {r['pnl (R$)']:,.0f} R$"
    texts = [label(r) for _, r in tf.iterrows()]
    date = lambda s: s.dt.strftime("%Y-%m-%d").tolist()
    return [
        (go.Scatter(x0=price["x0"], dx=price["dx"], y=price["y"].round(1), name="price (rebased to 100)", line={"color": "#555", "width": 1}), 1),
        (go.Scatter(x=date(tf["entry"]), y=(100 * tf["avg entry"] / base).round(1), mode="markers", name="entry", marker={"symbol": "triangle-up", "size": 8, "color": "#1f77b4"}, text=texts), 1),
        (go.Scatter(x=date(tf["exit"]), y=(100 * tf["exit price"] / base).round(1), mode="markers", name="exit", marker={"symbol": "x", "size": 8, "color": ["#2ca02c" if w else "#d62728" for w in wins]}, text=texts), 1),
        (go.Scatter(x0=equity["x0"], dx=equity["dx"], y=equity["y"].round(0), name="equity (R$)", line={"color": "#1f77b4"}), 2),
        (go.Scatter(x0=drawdown["x0"], dx=drawdown["dx"], y=(-100 * drawdown["y"]).round(1), name="drawdown (%)", fill="tozeroy", line={"color": "#d62728", "width": 1}), 3),
        (go.Bar(x=date(tf["exit"]), y=(100 * tf["pnl / capital"]).round(1), name="trade result (% of capital)", marker_color=["#2ca02c" if w else "#d62728" for w in wins]), 4),
    ]


def ticker_figure(t: str) -> go.Figure:
    figure = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04, row_heights=[0.34, 0.24, 0.16, 0.26],
                           subplot_titles=("Price (rebased to 100, weekly) and trades", "Equity (R$)", "Drawdown (%)", "Result of each trade (% of capital)"))
    for trace, row in ticker_traces(t):
        figure.add_trace(trace, row=row, col=1)
    figure.update_layout(title=f"{t}: Turtle, next_open", height=860, showlegend=False, hovermode="x unified")
    return figure


def best_and_worst(t: str, n: int = 3) -> pd.DataFrame:
    tf = trades_frame(t).drop(columns=["pnl / capital"])
    return pd.concat([tf.nlargest(n, "pnl (R$)").assign(which="best"), tf.nsmallest(n, "pnl (R$)").assign(which="worst")]).set_index("which")


for ticker in nine:
    print(f"\n==================== {ticker} ({group_of(ticker)}) ====================")
    ticker_figure(ticker).show()
    display(stats_for(ticker))
    display(best_and_worst(ticker).round(2))


==================== BPAC11 (shortlist) ====================


,BPAC11
statistic,
Period,2017-02-24 to 2026-10-16 (2396 candles)
Closed trades,31
Win rate,29.0%
Average gain / average loss (R$),"17,753 / 1,210"
Payoff (gain / loss),14.67
Profit factor,6.00
Expected value per trade (% of capital),17.2%
Best / worst streak (trades),3 / 4
Net result (R$ and % of capital),"133,157 (532.6%)"


/tmp/ipykernel_419360/3078646739.py:84: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(best_and_worst(ticker).round(2))


,entry,exit,units,shares,avg entry,exit price,pnl (R$),days,reason
which,,,,,,,,,
best,2018-11-07,2019-08-27,4,12115,5.5900,12.5500,"84,226.4100",196,sell
best,2025-01-23,2025-07-22,4,2368,32.1500,40.0000,"18,542.9300",122,sell
best,2020-05-22,2020-08-18,4,2602,13.3200,20.2000,"17,885.4000",61,sell
worst,2022-08-05,2022-09-27,4,2085,25.6900,23.9300,"-3,694.7500",36,sell
worst,2020-12-16,2021-01-07,4,2456,23.2100,21.7700,"-3,556.5000",12,stop
worst,2025-09-08,2025-10-08,3,1415,47.9000,46.1400,"-2,526.6600",22,stop



==================== PRIO3 (shortlist) ====================


...

,entry,exit,units,shares,avg entry,exit price,pnl (R$),days,reason
which,,,,,,,,,
best,2026-01-15,2026-05-14,4,3137,32.7300,44.4600,"36,719.2200",80,sell
best,2017-12-28,2018-04-04,4,6017,16.5400,20.3100,"22,638.4100",65,sell
best,2016-03-02,2016-05-24,4,5310,6.3300,8.6900,"12,510.9900",57,sell
worst,2018-05-02,2018-05-24,4,2841,24.7300,20.0500,"-13,348.1400",16,stop
worst,2023-12-21,2024-03-11,4,2643,38.0100,35.6700,"-6,255.9100",52,sell
worst,2020-02-13,2020-02-26,2,1439,30.2900,27.0400,"-4,699.9700",7,stop



==================== VALE3 (reference) ====================


,VALE3
statistic,
Period,2012-01-03 to 2026-10-16 (3669 candles)
Closed trades,68
Win rate,14.7%
Average gain / average loss (R$),"13,035 / 1,383"
Payoff (gain / loss),9.43
Profit factor,1.63
Expected value per trade (% of capital),3.0%
Best / worst streak (trades),2 / 13
Net result (R$ and % of capital),"50,163 (200.7%)"


/tmp/ipykernel_419360/3078646739.py:84: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(best_and_worst(ticker).round(2))


,entry,exit,units,shares,avg entry,exit price,pnl (R$),days,reason
which,,,,,,,,,
best,2025-09-08,2026-03-06,4,2480,57.4100,80.9000,"58,164.4600",122,sell
best,2020-11-17,2021-02-03,4,1188,67.8300,88.8500,"24,921.3100",50,sell
best,2016-10-24,2016-12-23,4,2878,21.3100,25.4600,"11,888.9900",42,sell
worst,2019-01-10,2019-01-29,4,1289,51.9600,43.2700,"-11,244.4700",12,sell
worst,2018-09-19,2018-10-08,4,1432,60.1100,57.0000,"-4,501.8100",13,stop
worst,2024-09-25,2024-10-09,3,1133,63.4400,60.6700,"-3,180.5600",10,stop


## 10. Any ticker: pick one from the dropdown

All candidates with data, not only the nine. The dropdown above the chart switches the price chart, the equity curve, the drawdown, the trade results and the statistics table together. It is a Plotly dropdown, so it needs no extra package, but it works only in a Jupyter front end or in an HTML export.

In [15]:
choices = sorted(universe)
dropdown = make_subplots(
    rows=4, cols=2, column_widths=[0.62, 0.38], vertical_spacing=0.05, horizontal_spacing=0.03,
    specs=[[{"type": "xy"}, {"type": "table", "rowspan": 4}], [{"type": "xy"}, None], [{"type": "xy"}, None], [{"type": "xy"}, None]],
    subplot_titles=("Price (rebased to 100, weekly) and trades", "", "Equity (R$)", "Drawdown (%)", "Result of each trade (% of capital)"),
)
owners = []  # which ticker each trace belongs to
for ticker in choices:
    if ticker not in full.index:
        continue
    for trace, row in ticker_traces(ticker):
        trace.visible = ticker == choices[0]
        dropdown.add_trace(trace, row=row, col=1)
        owners.append(ticker)
    table = stats_for(ticker)
    dropdown.add_trace(
        go.Table(header={"values": ["statistic", ticker], "align": "left"}, cells={"values": [list(table.index), list(table[ticker])], "align": "left"}, visible=ticker == choices[0]),
        row=1, col=2,
    )
    owners.append(ticker)

shown = [t for t in choices if t in full.index]
buttons = [
    {"label": f"{t}  ({group_of(t)})", "method": "update",
     "args": [{"visible": [owner == t for owner in owners]}, {"title": f"{t}: Turtle, next_open"}]}
    for t in shown
]
dropdown.update_layout(
    title=f"{shown[0]}: Turtle, next_open", height=900, showlegend=False,
    updatemenus=[{"buttons": buttons, "direction": "down", "x": 0.0, "xanchor": "left", "y": 1.12, "yanchor": "top", "showactive": True}],
)
print(f"{len(shown)} tickers in the dropdown")
dropdown.show()

48 tickers in the dropdown


## 11. Realistic or optimistic fill, for the nine

`signal_close` is the script's own fill (every order at the signal candle's close, optimistic); `next_open` is the one used above. A large gap between the rows says the result leans on the optimistic fill.

In [16]:
rows = []
for fill in ("signal_close", "next_open"):
    scan = scan_one({t: universe[t] for t in nine}, replace(CONFIG, fill_at=fill), MIN_CANDLES)
    rows.append(pd.DataFrame({
        "fill": fill, "cagr": scan["cagr"], "excess": scan["excess"], "max_equity_drawdown": scan["max_equity_drawdown"],
        "n_trades": scan["n_trades"],
    }))
fills = pd.concat(rows).reset_index(names="ticker").pivot(index="ticker", columns="fill")
fills.loc[nine]

cagr                 excess              max_equity_drawdown  \
fill   next_open signal_close next_open signal_close           next_open   
ticker                                                                     
BPAC11    0.2109       0.1940    0.1190       0.1020              0.3944   
PRIO3     0.1716       0.1729    0.0736       0.0749              0.5853   
MGLU3     0.1615       0.1630    0.0635       0.0649              0.4660   
WEGE3     0.1386       0.1268    0.0405       0.0288              0.2998   
ENEV3     0.1149       0.1095    0.0169       0.0114              1.1286   
RADL3     0.1036       0.1031    0.0056       0.0051              0.3833   
RENT3     0.0981       0.0953    0.0001      -0.0028              0.4509   
PETR4     0.0321       0.0516   -0.0659      -0.0464              0.9207   
VALE3     0.0773       0.0679   -0.0207      -0.0301              1.2182   

                     n_trades               
fill   signal_close next_open signal_close  
ticker                                      
BPAC11       0.3985        31           31  
PRIO3        0.5000        58           58  
MGLU3        0.4404        47           45  
WEGE3        0.3988        53           56  
ENEV3        1.5555        48           47  
RADL3        0.3882        59           59  
RENT3        0.4842        57           60  
PETR4        0.7972        66           67  
VALE3        1.5132        68           68

## 12. What this notebook does not answer

- **Whether the nine will keep doing well.** They were chosen on this history. Section 8 is the honest measure of how much that matters, and `turtle_stocks.ipynb` found no reliable long-horizon persistence.
- **A realistic account.** The capital cap is a deferred rule (`docs/brief.md` section 10): the positions here need several times `capital_brl`, and some names lost more than all of it.
- **A fair comparison with fully invested benchmarks.** The CDI and the IBOV are in the market every day, the strategy is not. A same-days comparison needs an approved spec before any code.
- **Which signal wins when several fire together.** The slots are independent, with no shared cash.

Write what you conclude from the tables and charts above here, in your own words, before relying on any of them.